# Gaussian noise, and why Corollary 2 is conservative

`histogram_recovery.ipynb` is noiseless. The Streamlit app's noise tab adds error of a
*chosen norm* --- a clean probe of the paper's Corollary 2, but not a channel.

A real system that recovers a bundled hypervector does not hand you a perturbation of
some fixed size. It perturbs every entry independently, and roughly normally. That is
the model here:

$$s \;=\; \sum_{i=1}^{t} c_i \;+\; e, \qquad e_k \sim \mathcal{CN}(0, \sigma^2)
\ \text{ i.i.d.}$$

$\sigma$ is in units of the signal: a codeword entry has $\lvert\text{entry}\rvert = 1$,
so $\sigma = 1$ means noise as large as the thing it is corrupting.

Three things come out of it, and all three are checked below:

1. Stage A has an **exact closed form** under this model, not just a bound.
2. The sharp cliff at Corollary 2's threshold **disappears** --- block norms vary, so
   failure arrives gradually.
3. Corollary 2 gets **more conservative as $N$ grows**. At $N = 27$ the usable noise is
   about $2\times$ its threshold; by $N = 2048$ it is over $12\times$.

In [1]:
import sys
from math import erf, log, sqrt

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

sys.path.insert(0, "streamlit_app")
import core                      # the app's plain functions; no Sage needed for Stage A

%matplotlib inline
np.set_printoptions(precision=3, suppress=True, linewidth=200)

p, m, K = 3, 3, 3
ctx = core.numpy_context(p, m, K)
N, DIM = ctx["N"], ctx["dim"]
THRESHOLD = np.sqrt(N) / 2                       # Corollary 2, per block

MSGS = core.message_list(ctx, 3)
CLEAN = core.bundle_of(ctx, MSGS)
H_CLEAN = core.histograms(ctx, CLEAN)

print(f"p = {p}, m = {m}, K = {K}")
print(f"  N   = {N}      codeword length, and one block")
print(f"  dim = {DIM}     entries in the hypervector = N blocks of N")
print(f"  Corollary 2 permits ||e_j||_2 < sqrt(N)/2 = {THRESHOLD:.4f} per block")

p = 3, m = 3, K = 3
  N   = 27      codeword length, and one block
  dim = 729     entries in the hypervector = N blocks of N
  Corollary 2 permits ||e_j||_2 < sqrt(N)/2 = 2.5981 per block


## 1 --- The two noise models side by side

The app's probe fixes $\lVert e_j \rVert_2 = \alpha \sqrt{N}/2$ in *every* block.
Gaussian noise does not: each block's norm is a random draw. That single difference is
what turns a cliff into a curve.

In [2]:
sigma = 1.0
e_gauss = core.gaussian_noise(ctx, sigma, seed=0)
e_probe = core.noise_for(ctx, 2.0, kind="isotropic", seed=0)   # same mean norm

nb_g = core.block_norms(ctx, e_gauss)
nb_p = core.block_norms(ctx, e_probe)

print(f"per-block ||e_j||_2 over the {N} blocks:\n")
print(f"   {'model':<22} {'min':>8} {'mean':>8} {'max':>8} {'std':>8}")
print(f"   {'Gaussian, sigma=1.0':<22} {nb_g.min():>8.3f} {nb_g.mean():>8.3f} "
      f"{nb_g.max():>8.3f} {nb_g.std():>8.3f}")
print(f"   {'probe, alpha=2.0':<22} {nb_p.min():>8.3f} {nb_p.mean():>8.3f} "
      f"{nb_p.max():>8.3f} {nb_p.std():>8.3f}")
print(f"\n   the probe has std 0 by construction -- every block is identical.")
print(f"   Gaussian block norms vary, and sit here relative to the "
      f"threshold {THRESHOLD:.3f}:")
print(f"      {(nb_g > THRESHOLD).sum()} of {N} blocks ABOVE it, "
      f"{(nb_g <= THRESHOLD).sum()} below")
print(f"\n   and yet Stage A is exact: "
      f"{np.array_equal(core.histograms(ctx, CLEAN + e_gauss), H_CLEAN)}")
print(f"   so being over the Corollary 2 threshold is not, by itself, failure.")

per-block ||e_j||_2 over the 27 blocks:

   model                       min     mean      max      std
   Gaussian, sigma=1.0       4.291    5.126    6.126    0.485
   probe, alpha=2.0          5.196    5.196    5.196    0.000

   the probe has std 0 by construction -- every block is identical.
   Gaussian block norms vary, and sit here relative to the threshold 2.598:
      27 of 27 blocks ABOVE it, 0 below

   and yet Stage A is exact: True
   so being over the Corollary 2 threshold is not, by itself, failure.


## 2 --- Stage A under Gaussian noise, exactly

Recall what actually decides Stage A. Writing $e = Ha^\top + i\,Hb^\top$,

$$\tfrac{1}{N}\,\Re\!\left(H^\dagger s_j\right) \;=\; h_j + a,$$

so the rounding returns $h_j$ **iff every coordinate of $a$ lies in $(-0.5, 0.5)$**.
$b$ never appears.

Now put Gaussian noise in. $H$ has orthogonal columns with $\lVert \cdot \rVert^2 = N$,
so $(H^\dagger e)_i \sim \mathcal{CN}(0, N\sigma^2)$, and therefore

$$a_i \;=\; \tfrac{1}{N}\Re\!\left(H^\dagger e\right)_i \;\sim\;
\mathcal{N}\!\left(0,\ \tfrac{\sigma^2}{2N}\right) \quad\text{i.i.d.}$$

Note the $1/\sqrt{N}$: the transform **averages the noise down**. That is the processing
gain that makes $\sigma \approx 1$ survivable at all.

In [3]:
rows = []
for sg in [0.25, 0.5, 1.0, 2.0]:
    samples = []
    for seed in range(60):
        e = core.gaussian_noise(ctx, sg, seed=seed)
        blocks = e.reshape(N, N)
        a = (ctx["H"].conj().T @ blocks.T).real / N
        samples.append(a.ravel())
    a_all = np.concatenate(samples)
    rows.append((sg, sg / np.sqrt(2 * N), a_all.std(), a_all.mean()))

print(f"   {'sigma':>7} {'predicted std':>15} {'measured std':>14} {'measured mean':>15}")
for sg, pred, meas, mn in rows:
    print(f"   {sg:>7} {pred:>15.5f} {meas:>14.5f} {mn:>15.5f}")
print(f"\n   predicted std is sigma / sqrt(2N) = sigma / {np.sqrt(2 * N):.3f}")

     sigma   predicted std   measured std   measured mean
      0.25         0.03402        0.03390        -0.00015
       0.5         0.06804        0.06781        -0.00030
       1.0         0.13608        0.13561        -0.00060
       2.0         0.27217        0.27122        -0.00119

   predicted std is sigma / sqrt(2N) = sigma / 7.348


## 3 --- The closed form

Stage A is exact when **all** $N^2$ coordinates (one per block per symbol) stay in
range. They are independent, so

$$\boxed{\;P(\text{Stage A exact}) \;=\;
\operatorname{erf}\!\left(\frac{\sqrt{N}}{2\sigma}\right)^{\!N^2}\;}$$

No bound, no constant --- an equality. Compare it against Monte Carlo.

In [4]:
TRIALS = 400
print(f"   {'N':>5} {'sigma':>7} {'P closed form':>15} {'P measured':>12} {'trials':>8}")
for (pp, mm) in [(3, 2), (3, 3), (5, 2)]:
    c = core.numpy_context(pp, mm, 3)
    msgs = core.message_list(c, 3)
    s0 = core.bundle_of(c, msgs)
    h0 = core.histograms(c, s0)
    for sg in [0.6, 0.8, 1.0, 1.1, 1.3]:
        hit = sum(np.array_equal(core.histograms(c, s0 + core.gaussian_noise(c, sg, seed=i)),
                                 h0) for i in range(TRIALS))
        print(f"   {c['N']:>5} {sg:>7} {core.stage_a_probability(c['N'], sg):>15.4f} "
              f"{hit / TRIALS:>12.4f} {TRIALS:>8}")

       N   sigma   P closed form   P measured   trials
       9     0.6          0.9676       0.9800      400
       9     0.8          0.5213       0.5325      400
       9     1.0          0.0612       0.0850      400
       9     1.1          0.0113       0.0175      400
       9     1.3          0.0002       0.0025      400


      27     0.6          1.0000       1.0000      400
      27     0.8          0.9968       0.9900      400
      27     1.0          0.8404       0.8600      400
      27     1.1          0.5431       0.5750      400
      27     1.3          0.0320       0.0300      400
      25     0.6          1.0000       1.0000      400
      25     0.8          0.9938       0.9950      400


      25     1.0          0.7754       0.7925      400
      25     1.1          0.4412       0.4550      400
      25     1.3          0.0166       0.0150      400


## 4 --- Where the usable threshold actually sits

Define $\sigma_{1/2}$ as the noise level at which Stage A succeeds half the time. The
interesting comparison is against Corollary 2, whose guarantee is a *norm*
($\sqrt{N}/2$), so convert: Gaussian noise of level $\sigma$ gives a mean block norm of
about $\sigma\sqrt{N}$, hence a multiple $2\sigma$ of the threshold.

In [5]:
print(f"   {'N':>6} {'sigma_1/2':>11} {'mean ||e_j||':>13} {'x Cor. 2 threshold':>20}")
for Nv in [4, 9, 16, 25, 27, 49, 81, 121, 343, 625, 2048]:
    sh = core.sigma_half(Nv)
    print(f"   {Nv:>6} {sh:>11.4f} {sh * np.sqrt(Nv):>13.3f} {2 * sh:>19.2f}x")

print(f"\n   Corollary 2 is sufficient, never necessary -- and the gap WIDENS with N.")
print(f"   At N = 27 you can run at ~2x its threshold; at N = 2048, over 12x.")

        N   sigma_1/2  mean ||e_j||   x Cor. 2 threshold
        4      0.6968         1.394                1.39x
        9      0.8064         2.419                1.61x
       16      0.9430         3.772                1.89x
       25      1.0840         5.420                2.17x
       27      1.1118         5.777                2.22x
       49      1.3653         9.557                2.73x
       81      1.6414        14.772                3.28x
      121      1.9119        21.030                3.82x
      343      2.8908        53.537                5.78x
      625      3.7002        92.504                7.40x
     2048      6.1131       276.646               12.23x

   Corollary 2 is sufficient, never necessary -- and the gap WIDENS with N.
   At N = 27 you can run at ~2x its threshold; at N = 2048, over 12x.


### Why the gap widens

$P = \operatorname{erf}(x)^{N^2}$ with $x = \sqrt{N}/2\sigma$. Using
$1 - \operatorname{erf}(x) \approx e^{-x^2}/(x\sqrt{\pi})$, the product stays near $1$
while $e^{-x^2} \lesssim N^{-2}$, i.e. $x \gtrsim \sqrt{2\log N}$. So

$$\sigma_{1/2} \;\asymp\; \frac{\sqrt{N}}{2\sqrt{2\log N}} \;\propto\;
\sqrt{\frac{N}{\log N}}.$$

Two effects race: the transform averages noise over $N$ entries (gain $\sqrt{N}$),
while the number of coordinates that must all stay in range grows as $N^2$ (cost
$\sqrt{\log N}$). The gain wins. **Bigger fields are more noise-robust**, and
Corollary 2's fixed $\sqrt{N}/2$ cannot see that because it throws away both the
imaginary half of the noise and the difference between $\lVert a\rVert_\infty$ and
$\lVert a\rVert_2$.

In [6]:
print(f"   {'N':>6} {'sigma_1/2':>11} {'sqrt(N/log N)':>15} {'ratio':>8}")
for Nv in [9, 27, 81, 343, 625, 2048, 8192]:
    sh, sc = core.sigma_half(Nv), np.sqrt(Nv / log(Nv))
    print(f"   {Nv:>6} {sh:>11.4f} {sc:>15.4f} {sh / sc:>8.4f}")
print(f"\n   the ratio is near-constant, so the sqrt(N/log N) law holds.")

        N   sigma_1/2   sqrt(N/log N)    ratio
        9      0.8064          2.0239   0.3984
       27      1.1118          2.8622   0.3884
       81      1.6414          4.2933   0.3823
      343      2.8908          7.6652   0.3771
      625      3.7002          9.8531   0.3755
     2048      6.1131         16.3891   0.3730
     8192     11.1786         30.1516   0.3707

   the ratio is near-constant, so the sqrt(N/log N) law holds.


## 5 --- The picture: a curve, not a cliff

Left: $P(\text{Stage A exact})$ against $\sigma$, closed form as lines, Monte Carlo as
dots. Right: the tolerable noise against $N$, with Corollary 2's guarantee for
comparison.

In [ ]:
mpl.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "savefig.facecolor": "#fcfcfb", "text.color": "#0b0b0b",
    "axes.labelcolor": "#52514e", "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.edgecolor": "#dcdbd6", "grid.color": "#dcdbd6",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.linewidth": 0.6, "grid.alpha": 0.9,
    "lines.linewidth": 2.0, "font.size": 10, "axes.titlesize": 11,
    "axes.titleweight": "bold", "legend.frameon": False,
    "figure.dpi": 140, "savefig.dpi": 140,
})
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
import os
os.makedirs("figures", exist_ok=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.4, 4.6))

CASES = [(3, 2), (5, 2), (3, 3), (3, 4)]   # ascending N
grid = np.linspace(0.3, 3.2, 180)
for colour, (pp, mm) in zip(PALETTE, CASES):
    c = core.numpy_context(pp, mm, 3)
    Nv = c["N"]
    ax1.plot(grid, [core.stage_a_probability(Nv, g) for g in grid],
             color=colour, label=f"N = {Nv}")
    if Nv <= 81:                                  # Monte Carlo where it is quick
        msgs = core.message_list(c, 3)
        s0 = core.bundle_of(c, msgs)
        h0 = core.histograms(c, s0)
        xs = np.linspace(0.6, 2.4, 7)
        ys = [sum(np.array_equal(
                  core.histograms(c, s0 + core.gaussian_noise(c, g, seed=i)), h0)
                  for i in range(120)) / 120 for g in xs]
        ax1.plot(xs, ys, "o", color=colour, markersize=5, markeredgecolor="#fcfcfb",
                 markeredgewidth=1.2, linestyle="none")
ax1.set_xlabel(r"noise level  $\sigma$   (signal entries have $|$entry$| = 1$)")
ax1.set_ylabel("P(Stage A returns every histogram exactly)")
ax1.set_title("Gaussian noise degrades smoothly")
ax1.legend(loc="upper right")
ax1.annotate("lines: closed form\ndots: Monte Carlo", xy=(0.42, 0.30), fontsize=9,
             color="#52514e")

Ns = np.array([4, 9, 16, 25, 27, 49, 64, 81, 121, 243, 343, 625, 1024, 2048])
sig_half = np.array([core.sigma_half(int(v)) for v in Ns])
ax2.plot(Ns, sig_half * np.sqrt(Ns), color=PALETTE[0], marker="o",
         label=r"usable: mean $\|e_j\|$ at $\sigma_{1/2}$")
ax2.plot(Ns, np.sqrt(Ns) / 2, color=PALETTE[1], marker="o",
         label=r"Corollary 2 guarantee: $\sqrt{N}/2$")
ax2.set_xscale("log"); ax2.set_yscale("log")
ax2.set_xlabel("N")
ax2.set_ylabel(r"block noise norm  $\|e_j\|_2$")
ax2.set_title("The gap widens with N")
ax2.legend(loc="upper left")

fig.tight_layout()
fig.savefig("figures/gaussian_stage_a.png", bbox_inches="tight")
plt.show()

print(f"at N = 27 the usable level is {2 * core.sigma_half(27):.2f}x the guarantee;"
      f"  at N = 2048 it is {2 * core.sigma_half(2048):.2f}x.")

## 6 --- End to end: does *recovery* survive?

Stage A exactness is not the same as recovering the bundle. Stage B has slack of its
own --- a damaged histogram can still decode, because the Reed--Solomon list decoder
only needs each true codeword to score the maximum. This needs Sage, so it is the one
slow section.

In [8]:
sage_ctx = core.build_context(p, m, K)      # the Sage half; a second or two
U_MAX = sage_ctx["u_max"]
print(f"u_max = {U_MAX} for p={p}, m={m}, K={K}\n")

print(f"   {'t':>4} {'sigma':>7} {'Stage A':>9} {'returned':>9} {'exact?':>8}")
for t in [1, 3, U_MAX]:
    msgs = core.message_list(sage_ctx, t)
    s0 = core.bundle_of(sage_ctx, msgs)
    h0 = core.histograms(sage_ctx, s0)
    for sg in [0.0, 0.8, 1.1, 1.5]:
        e = core.gaussian_noise(sage_ctx, sg, seed=0) if sg else 0
        res = core.decode(sage_ctx, s0 + e, d=1, ell_max=min(40, U_MAX + 4))
        a_ok = np.array_equal(res["hist"], h0)
        print(f"   {t:>4} {sg:>7} {('exact' if a_ok else 'WRONG'):>9} "
              f"{len(res['messages']):>9} {str(res['messages'] == sorted(msgs)):>8}")

u_max = 13 for p=3, m=3, K=3

      t   sigma   Stage A  returned   exact?
      1     0.0     exact         1     True
      1     0.8     exact         1     True
      1     1.1     WRONG         1     True
      1     1.5     WRONG         0    False
      3     0.0     exact         3     True
      3     0.8     exact         3     True
      3     1.1     WRONG         3     True


      3     1.5     WRONG         2    False


     13     0.0     exact        13     True


     13     0.8     exact        13     True


     13     1.1     WRONG         0    False


     13     1.5     WRONG         0    False


## Summary

| | the app's probe | Gaussian (this notebook) |
|---|---|---|
| what is fixed | $\lVert e_j\rVert_2$, identically in every block | $\sigma$ per entry; block norms vary |
| failure shape | a cliff at $\alpha = 1$ | a smooth curve |
| what it answers | is Corollary 2 tight? (yes, in the worst direction) | what does a real channel do? |
| Stage A | pass / fail | $\operatorname{erf}(\sqrt{N}/2\sigma)^{N^2}$, exact |

The two are complementary, and neither alone is the whole story. The probe shows the
bound cannot be improved *as a worst-case statement over directions*. This notebook
shows that for the noise an actual system produces, the bound is conservative by a
factor that grows like $\sqrt{N/\log N}$ --- a system at $N = 625$ can run at over
$7\times$ the norm Corollary 2 permits and still recover its histograms.